# 01 · `recent-rate-v1` — what the stage does today

```
reserve = (burn in the last L minutes / L) x minutes_to_window_end x multiplier
```

Two knobs: the look-back `L` and the `multiplier`. Today they are 30 minutes and 1.5,
chosen by reasoning rather than measurement — so tuning them by cost is the first
fair thing anyone has done to this engine.

Its structural defect is that it has no idea what time it is: 2 a.m. and 2 p.m. are
forecast identically from identical recent activity.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd() / "lab"))
import numpy as np, pandas as pd
import dataset as D

frame = D.build("claude")
train, validation, test = D.split(frame)
COLS = D.features(frame)
print("\ncost model:  stolen $%.2f/pct   wasted $%.2f/pct   ratio %.0f:1"
      % (D.COST_STOLEN, D.COST_WASTED, D.COST_STOLEN / D.COST_WASTED))
print("the cost-optimal quantile is therefore %.4f" % D.optimal_quantile())

12075 anchors over 41.9 days, embargo 300 min at each seam
  train        7185 rows  08-24 17:55 -> 09-18 16:35
  validation   2355 rows  09-18 21:40 -> 09-27 01:50
  test         2415 rows  09-27 06:55 -> 10-05 16:05
  dropped to the embargo: 120

cost model:  stolen $3.20/pct   wasted $0.32/pct   ratio 10:1
the cost-optimal quantile is therefore 0.9091


## The objective

Not accuracy — **money**. Two unit costs per percent of a 5-hour meter:

| | $/pct | Meaning |
|---|---|---|
| `COST_WASTED` | **0.32** | quota the bot was refused although the human never came — exactly 1% of a $32 unit |
| `COST_STOLEN` | **3.20** | quota the human wanted and the bot took — 10×, because the user has first claim |

Hyper-parameters below are chosen on **validation**, by cost. The test set is read
only in `05_comparison.ipynb`.

In [2]:
# Only the 30-minute arm is reproducible here: this table carries no window, by
# design. That is the arm the live stage uses whenever it is the larger of the two.
def recent_rate(f, N, lookback, multiplier):
    return f["burn_%dm" % lookback].values / lookback * N * multiplier

candidates = {"L=%d x%.2f" % (L, m): (L, m)
              for L in (5, 15, 30, 60, 120, 240)
              for m in (0.5, 1.0, 1.5, 2.0, 3.0, 5.0)}
chosen, table = D.tune("recent-rate", candidates, validation,
                       lambda c, N: recent_rate(validation, N, c[0], c[1]), show=4)

N = 30   best: L=120 x2.00                  $0.4513   (worst $0.8499)
     config  cost_usd  stolen_usd  wasted_usd
L=120 x2.00    0.4513      0.3254      0.1259
 L=60 x1.50    0.4563      0.3713      0.0850
 L=30 x1.50    0.4635      0.3546      0.1088
 L=30 x2.00    0.4650      0.2772      0.1878

N = 60   best: L=30 x2.00                   $1.0614   (worst $1.8396)
    config  cost_usd  stolen_usd  wasted_usd
L=30 x2.00    1.0614      0.6332      0.4282
L=30 x3.00    1.1183      0.3737      0.7446
L=30 x1.50    1.1381      0.8574      0.2807
L=60 x2.00    1.1593      0.7270      0.4324

N = 120  best: L=60 x3.00                   $2.3656   (worst $4.8603)
    config  cost_usd  stolen_usd  wasted_usd
L=60 x3.00    2.3656      1.0476      1.3179
L=30 x3.00    2.4094      1.1740      1.2354
L=60 x2.00    2.4207      1.5667      0.8540
L=30 x2.00    2.4456      1.5762      0.8694

N = 180  best: L=30 x5.00                   $4.6878   (worst $8.0794)
    config  cost_usd  stolen_usd  was

In [3]:
print("today's hand-picked L=30, multiplier 1.5 -- and what tuning prefers instead:")
today = table[table["config"] == "L=30 x1.50"].set_index("N")["cost_usd"]
for N in D.HORIZONS:
    bestc = table[(table["N"] == N) & (table["config"] == chosen[N][0])]["cost_usd"].iloc[0]
    print("  N=%-4d today $%.4f   tuned $%.4f  (%s)  %+.1f%%"
          % (N, today[N], bestc, chosen[N][0], (bestc - today[N]) / today[N] * 100))

today's hand-picked L=30, multiplier 1.5 -- and what tuning prefers instead:
  N=30   today $0.4635   tuned $0.4513  (L=120 x2.00)  -2.6%
  N=60   today $1.1381   tuned $1.0614  (L=30 x2.00)  -6.7%
  N=120  today $2.5441   tuned $2.3656  (L=60 x3.00)  -7.0%
  N=180  today $5.3541   tuned $4.6878  (L=30 x5.00)  -12.4%
  N=300  today $11.0005   tuned $9.8283  (L=30 x5.00)  -10.7%


In [4]:
val = [D.evaluate("recent-rate", validation["y%d" % N].values,
                  recent_rate(validation, N, *chosen[N][1])) | {"N": N} for N in D.HORIZONS]
print(pd.DataFrame(val).set_index("N")[["cost_usd", "stolen", "wasted", "coverage"]].round(3))
print()
print(D.by_cell(validation, validation["y300"].values,
                {"recent-rate": recent_rate(validation, 300, *chosen[300][1])}
                ).to_string(index=False))
D.save("01_recent_rate", {"kind": "recent_rate",
                          "per_horizon": {str(N): {"lookback": chosen[N][1][0],
                                                   "multiplier": chosen[N][1][1]}
                                          for N in D.HORIZONS}}, val)

     cost_usd  stolen  wasted  coverage
N                                      
30      0.451   0.102   0.393    95.074
60      1.061   0.198   1.338    91.847
120     2.366   0.327   4.118    88.917
180     4.688   0.877   5.876    84.459
300     9.828   2.430   6.416    77.028

 state      band   n  ACTUAL p95  recent-rate
  idle     night 759         5.0          0.0
  idle   morning 421        38.0          0.0
  idle afternoon 253        40.0          0.0
  idle   evening 479         0.0          0.0
  warm     night  20         3.0          0.0
  warm   morning  28        83.0          0.0
  warm afternoon  62        84.9          0.0
  warm   evening  92         0.0          0.0
active     night  12         3.0        100.0
active   morning  31        79.0         91.9
active afternoon 165        84.0         94.5
active   evening  33         5.4         90.9
froze 01_recent_rate -> design/02_prediction/lab/results/01_recent_rate.json


PosixPath('/Users/jeanlescut/dev/agent-quota-maximizer/design/02_prediction/lab/results/01_recent_rate.json')